In [ ]:
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.question_answering import load_qa_chain
import google.generativeai as genai
import warnings
warnings.filterwarnings('ignore')


In [ ]:
pdf = "/content/resume.pdf"
pdf_reader = PdfReader(pdf)
print(pdf_reader)


In [ ]:
# Extract text from each page separately
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

print(text)


In [ ]:
# Split the long text into small chunks.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
    length_function=len)

chunks = text_splitter.split_text(text=text)
print(f"Number of chunks: {len(chunks)}")


In [ ]:
chunks[0]


In [ ]:
chunks[1]


The above text is common(overlap) for both chunks[0] and chunks[1].
(chunk_overlap=200 - maximum length, it means length is not exceed 200)


In [ ]:
gemini_api_key = input('Enter your Google Gemini API Key: ')


In [ ]:
def gemini(gemini_api_key, chunks, analyze):

    # Configure Google Generative AI
    genai.configure(api_key=gemini_api_key)
    model = genai.GenerativeModel("models/gemini-2.5-flash")

    # Using Google Gemini service for embedding
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=gemini_api_key)

    # Facebook AI Similarity Search library help us to convert text data to numerical vector
    vectorstores = FAISS.from_texts(chunks, embedding=embeddings)

    # compares the query and chunks, enabling the selection of the top 'K' most similar chunks based on their similarity scores.
    docs = vectorstores.similarity_search(query=analyze, k=3)

    # Prepare context from retrieved documents
    context = "\n".join([doc.page_content for doc in docs])
    
    # Create prompt with context and question
    prompt = f"""Based on the following context, please answer the question:

Context:
{context}

Question: {analyze}

Please provide a detailed and helpful response based on the context provided."""

    # Generate response using Gemini 1.5 Flash
    response = model.generate_content(prompt)
    return response.text


In [ ]:
def resume_summary(query_with_chunks):
    query = f''' need to detailed summarization of below resume and finally conclude them

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

summary = resume_summary(query_with_chunks=chunks)
summary_result = gemini(gemini_api_key=gemini_api_key, chunks=chunks, analyze=summary)
print(summary_result)


In [ ]:
def resume_strength(query_with_chunks):
    query = f''' need to detailed analysis and explain of the strength of below resume.

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

strength = resume_strength(query_with_chunks=summary_result)
strength_result = gemini(gemini_api_key=gemini_api_key, chunks=chunks, analyze=strength)
print(strength_result)


In [ ]:
def resume_weakness(query_with_chunks):
    query = f'''need to detailed analysis and explain of the weakness of below resume and how to improve make a better resume.

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

weakness = resume_weakness(query_with_chunks=summary_result)
result_weakness = gemini(gemini_api_key=gemini_api_key, chunks=chunks, analyze=weakness)
print(result_weakness)


In [ ]:
def job_title_suggestion(query_with_chunks):

    query = f''' what are the job roles i apply to linkedin based on below?
                  
                  """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                  {query_with_chunks}
                  """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

suggestion = job_title_suggestion(query_with_chunks=summary_result)
result_suggestion = gemini(gemini_api_key=gemini_api_key, chunks=chunks, analyze=suggestion)
print(result_suggestion)
